In [1]:
from rdkit import Chem
import numpy as np
from torch.utils.data import DataLoader

from chemprop.data import (
    MoleculeDatapoint,
    MoleculeDataset,
    MulticomponentDataset,
)
from chemprop.featurizers.molecule import ChargeFeaturizer
from chemprop.nn.agg import MeanAggregation
from chemprop.nn.message_passing import BondMessagePassing, MulticomponentMessagePassing
from chemprop.nn.predictors import RegressionFFN

from chemprop_contrib.mixtures.data import (
    InteractionDatapoint,
    InteractionDataset,
    MixtureDatapoint,
    MixtureDataset,
    collate_interaction_batch,
    collate_multicomponent_with_mixture,
)
from chemprop_contrib.mixtures.featurizers import CompleteInteractionGraphFeaturizer
from chemprop_contrib.mixtures.id import (
    IDMixtureDatapoint,
    IDMixtureDataset,
    IDMoleculeDatapoint,
    IDMoleculeDataset,
    MolGraphStore,
)
from chemprop_contrib.mixtures.models import InteractionMPNN, MixtureMPNN
from chemprop_contrib.mixtures.nn import (
    MixtureMessagePassing,
    NoMessagePassing,
    WeightedSumAggregation,
)

### Mixtures and Interactions

This module adds two new functionalities to chemprop:
1. Order invariant aggregation of learned molecule representations (here termed "fingerprints"), which accept any number of input molecules and give a fixed length output.
2. Message passing between molecule fingerprints, where the fingerprints act as nodes and interactions between them act as edges.

### Mixtures

``MixtureDatapoint``s are the same as ``MoleculeDatapoint``s except the former takes a list of molecules in the mixture instead of a single molecule and also takes an optional list of weights to apply to the fingerprints before aggregation. These weights could be the component's mole fraction, for example.

Similarly a ``MixtureDataset`` is the same as a ``MoleculeDataset`` except it takes ``MixtureDatapoint``s.

Base Chemprop uses simple concatenation of fingerprints when given multiple molecules. This works when the molecules are distinguishable, e.g., solute + solvent, but lacks flexibility for molecules in a mixture, e.g., binary solvent. The following "mixture aggregators" can be used for combining the fingerprints of molecules in a mixture into a single learned representation of the mixture prior to Chemprop's simple concatenation. These classes are based on `chemprop_contrib.mixtures.nn.agg.MixtureAggregation`, which differs from `chemprop.nn.agg.Aggregation` in that it can apply pre-determined weights to the elements being aggregated.

* ``WeightedSumAggregation``: Mixture-wise sum of molecular fingerprints multiplied by their individual weights.
* ``DeepsetsAggregation``: Can be seen as an extension of ``WeightedSumAggregation``, whereas the individual molecular fingerprints multiplied by the weights pass a _local_ MLP before being summed and then the mixture-wise sums pass a _global_ MLP.
* ``AttentiveAggregation``: Mixture-wise attention layer applied to molecular fingerprints multiplied by their individual weights (meaning that the weighting of the individual fingerprints is adjusted by attention logits).
* ``Set2SetAggregation``: Recurrent architecture based on LSTMs that aggregate mixture-wise molecular fingerprints multiplied by their individual composition into a mixture representation.

We also provide ``ConcatAggregation`` which simply concatenates all molecular fingerprints and the individual weights, but also adds zero-padding when less than the maximum number of molecules is provided, ensuring the mixture representation is fixed-length. This class can also randomize the order of the input fingerprints and zero-padding to help learn some order invariance. 

The mixture aggregator is given to ``MixtureMPNN`` as shown below.

In [2]:
solute_mp = BondMessagePassing()
solvent_mp = BondMessagePassing()
mcmp = MulticomponentMessagePassing(blocks=[solute_mp, solvent_mp], n_components=2)
agg = MeanAggregation()
mixture_agg = WeightedSumAggregation(solvent_mp.output_dim)
ffn = RegressionFFN(input_dim=(solute_mp.output_dim + mixture_agg.output_dim))

model1 = MixtureMPNN(mcmp, agg, mixture_agg, ffn)

/home/knathan/anaconda3/envs/chemprop_contrib/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'mixture_agg' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['mixture_agg'])`.


### Interactions

After each molecule's graph passes through atom-to-molecule aggregation, the resulting fingerprints can be used to update each other in another round of message passing, where the molecules act as the nodes (a super-graph, or graph-of-graphs). 

While the fingerprints will automatically populate the interaction graph's node representations, extra molecule (node) and interaction (edge) information can be supplied via ``InteractionDatapoints`` analogous to extra atom and bond information in a ``MoleculeDatapoint``. 

The connections in the interaction graph are determined by the featurizer of an ``InteractionDataset``. The default is a complete graph with self loops. 

Any message passing class can be applied to the interaction graph. We provide the following message passing classes:

* ``InteractionMessagePassing``: This is identical to Chemprop's ``BondMessagePassing``.
* ``MolecularMessagePassing``: This is identical to Chemprop's ``AtomMessagePassing``.
* ``MixtureMessagePassing``: Simple message passing between connected nodes. No edge information is used. 

The interaction message passing module is given to ``InteractionMPNN`` as shown below:

In [3]:
solute_mp = BondMessagePassing()
solvent_mp = BondMessagePassing()
mcmp = MulticomponentMessagePassing(blocks=[solute_mp, solvent_mp], n_components=2)
agg = MeanAggregation()
interaction_mp = MixtureMessagePassing(d_v=solvent_mp.output_dim)
mixture_agg = WeightedSumAggregation(interaction_mp.output_dim)
ffn = RegressionFFN(input_dim=(solute_mp.output_dim + mixture_agg.output_dim))

model2 = InteractionMPNN(mcmp, agg, interaction_mp, mixture_agg, ffn)

/home/knathan/anaconda3/envs/chemprop_contrib/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'interaction_mp' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['interaction_mp'])`.


### Simple Complete Examples

In [4]:
solute_smiles = ["C", "CC", "CCC"]
solvent_smiless = [["O", "CCO"], ["CCCCC"], ["N", "CO", "COC"]]
w_fpss = [[0.5, 0.5], [1.0], [0.1, 0.2, 0.7]]
ys = [[1.0], [2.0], [3.0]]

dp_solutes = [MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(solute_smiles, ys)]
dp_solvents = [MixtureDatapoint.from_smis(smis, w_fps=w_fps) for smis, w_fps in zip(solvent_smiless, w_fpss)]

dset_solute = MoleculeDataset(dp_solutes)
dset_solvent = MixtureDataset(dp_solvents)

dataset = MulticomponentDataset([dset_solute, dset_solvent])

loader = DataLoader(dataset, collate_fn=collate_multicomponent_with_mixture)

# You can then make a lightning trainer and train model1 from above.

``InteractionDataset`` is a hybrid between ``MoleculeDataset`` and ``MulticomponentDataset``. When it is used, the targets go in it instead of the first subgraph dataset.

In [5]:
dp_interaction = [InteractionDatapoint(y) for y in ys]
dataset = InteractionDataset(dp_interaction, subgraph_datasets=[dset_solute, dset_solvent])

loader = DataLoader(dataset, collate_fn=collate_interaction_batch)

# You can then make a lightning trainer and train model2 from above.

### Extra molecule features

Base Chemprop v2 doesn't have a way to augment the fingerprints directly. Instead extra molecule features are treated as "datapoint descriptors", which are inserted just before the predictor. This approach doesn't work for mixtures because that information should be inserted before interaction message passing and because the number of molecules (and therefore extra features) is variable. 

Extra molecule features act as extra node features in the interaction datapoints. This can be done even if interaction message passing is not desired by using ``NoMessagePassing`` as shown below.

The featurizer of an ``InteractionDataset`` can also use custom molecule and interaction featurizers, similar to ``SimpleMoleculeMolGraphFeaturizer``. 

In [6]:
# Nodes in interaction graph follow the order the molecules appear in the subgraph datasets. 
molss = [[Chem.MolFromSmiles(smi) for smi in [solute_smi] + solvent_smis] 
         for solute_smi, solvent_smis in zip(solute_smiles, solvent_smiless)]

my_molecule_featurizer = ChargeFeaturizer()

pregenerate = False
if pregenerate:
    # Either use your molecule featurizer to pregenerate the features and save them in the datapoints
    V_fs = [np.concatenate([my_molecule_featurizer(mol) for mol in mols]) for mols in molss]
    dp_interaction = [InteractionDatapoint(y, V_f=V_f) for y, V_f in zip(ys, V_fs)]
    interaction_graph_featurizer = CompleteInteractionGraphFeaturizer(extra_mol_fdim=len(my_molecule_featurizer))
    dataset = InteractionDataset(dp_interaction, interaction_graph_featurizer, subgraph_datasets=[dset_solute, dset_solvent])
else:
    # Or give the interaction graph featurizer your molecule featurizer to have them computed automatically
    dp_interaction = [InteractionDatapoint(y) for y in ys]
    interaction_graph_featurizer = CompleteInteractionGraphFeaturizer(mol_featurizer=my_molecule_featurizer)
    dataset = InteractionDataset(dp_interaction, interaction_graph_featurizer, subgraph_datasets=[dset_solute, dset_solvent])

solute_mp = BondMessagePassing()
solvent_mp = BondMessagePassing()
mcmp = MulticomponentMessagePassing(blocks=[solute_mp, solvent_mp], n_components=2)
interaction_mp = NoMessagePassing(d_v=solvent_mp.output_dim)

### ID Data

As datasets of mixture tend to have many more datapoints than unique molecules, one possible way to cut down on computations is to compute all the ``MolGraph``s (featurized molecule graphs) once for each unique molecule at the beginning. This is possible using a ``MolGraphStore``. This class assigns each unique molecule (based on unique SMILES string, so canonicalization is useful) an ID number and creates dictionaries of ID to SMILES string, ID to ``Chem.Mol``, and ID to ``MolGraph``. Then ``IDMoleculeDatapoint`` takes these IDs instead of molecules.

If you use InChI's, you'll need to convert to SMILES string, which is what base Chemprop prefers. If you already have ``Chem.Mol`` objects, you'll still need to make SMILES strings as ``Chem.Mol`` objects don't hash reliably. 

In [7]:
smiles = solute_smiles + [smi for smis in solvent_smiless for smi in smis]
mg_store = MolGraphStore(smiles_strings=smiles)

dp_solutes = [IDMoleculeDatapoint(mol_id=mg_store.smiles_to_id[smi], y=y) for smi, y in zip(solute_smiles, ys)]
dp_solvents = [IDMixtureDatapoint(mol_ids=[mg_store.smiles_to_id[smi] for smi in smis]) for smis in solvent_smiless]

dset_solute = IDMoleculeDataset(dp_solutes, mg_store=mg_store)
dset_solvent = IDMixtureDataset(dp_solvents, mg_store=mg_store)

dataset = MulticomponentDataset([dset_solute, dset_solvent])

### Only Mixture 

If your dataset has only a mixture, e.g., viscosity of hydrocarbon blend, you still need to use the multicomponent framework.

In [8]:
blends = [["CCC", "C(C)CC"], ["CCCCC"], ["CCC(C)C", "CCO", "COCC"]]
w_fpss = [[0.5, 0.5], [1.0], [0.1, 0.2, 0.7]]
ys = [[1.0], [2.0], [3.0]]

dps = [MixtureDatapoint.from_smis(smis, w_fps=w_fps) for smis, w_fps in zip(blends, w_fpss)]

mixture_dset = MixtureDataset(dps)

dataset = MulticomponentDataset([mixture_dset])

loader = DataLoader(dataset, collate_fn=collate_multicomponent_with_mixture)

mcmp = MulticomponentMessagePassing(blocks=[BondMessagePassing()], n_components=1)
agg = MeanAggregation()
mixture_agg = WeightedSumAggregation(mcmp.output_dim)
ffn = RegressionFFN(input_dim=mixture_agg.output_dim)

model = MixtureMPNN(mcmp, agg, mixture_agg, ffn)